In [ ]:
!pip install -q "protobuf==5.29.6" kaggle-benchmarks
import kaggle_benchmarks as kbench

In [ ]:
import kaggle_benchmarks as kbench
import typing

# Mock implementations for self-contained execution
class MockStore(typing.NamedTuple):
    timeline: dict[tuple[str, str], list[tuple[int, str]]]

def build_store(version: str = "v1", seed: int = 0) -> MockStore:
    """Mocks the data store for demonstration."""
    return MockStore(
        timeline={
            ("USA", "President"): [
                (44, "Barack Obama is the 44th president."),
                (45, "Donald Trump is the 45th president."),
                (46, "Joe Biden is the 46th president."),
            ],
            ("UK", "Prime Minister"): [
                (2019, "Boris Johnson is the Prime Minister."),
                (2022, "Liz Truss is the Prime Minister."),
                (2023, "Rishi Sunak is the Prime Minister."),
            ],
        }
    )

def load_questions(version: str = "v1", seed: int = 0) -> list[dict]:
    """Mocks the question loader for demonstration."""
    return [
        {
            "task_family": "AsOfQA",
            "domain": "USA",
            "subject": "President",
            "as_of_day": 45,
            "prompt": "As of the end of the 45th presidency, who is the President of the USA?",
        },
        {
            "task_family": "PastQueryTrap",
            "domain": "USA",
            "subject": "President",
            "as_of_day": 47,
            "prompt": "Who is the President of the USA?",
        },
        {
            "task_family": "AsOfQA",
            "domain": "UK",
            "subject": "Prime Minister",
            "as_of_day": 2020,
            "prompt": "In the year 2020, who was the Prime Minister of the UK?",
        },
        {
            "task_family": "SimpleQA", # This should be filtered out
            "prompt": "What is the capital of France?",
        },
    ]

@kbench.task(name="StalenessDetection", description="Is this fact stale or current?")
def staleness_task(llm, version: str = "v1", seed: int = 0) -> None:
    """StalenessDetection: Binary classification — is the queried fact stale?"""
    store = build_store(version=version, seed=seed)
    questions = load_questions(version=version, seed=seed)

    asof_questions = [
        q
        for q in questions
        if q.get("task_family") in ("AsOfQA", "PastQueryTrap")
    ]

    for q in asof_questions:
        domain = q.get("domain", "")
        subject = q.get("subject", "")
        as_of_day = q.get("as_of_day", 50)

        if not domain or not subject:
            continue

        key = (domain, subject)
        latest_day = None
        if key in store.timeline:
            latest_day = max(day for day, _ in store.timeline[key])

        stale = latest_day is not None and latest_day > as_of_day

        prompt = f"Is this fact stale or current? {q.get('prompt', '')}"
        response = llm.prompt(prompt).strip()

        resp_lower = response.lower()
        llm_says_stale = "stale" in resp_lower or "outdated" in resp_lower or "old" in resp_lower
        correct = llm_says_stale == stale

        kbench.assertions.assert_true(
            correct,
            expectation=(
                f"[Staleness] domain={domain} subject={subject} "
                f"as_of_day={as_of_day} latest_day={latest_day} stale={stale} | "
                f"LLM: {response}"
            ),
        )

staleness_task.run(kbench.llm)